# 14. Deep Learning Teaser and Course Wrap-up

We've come a long way. In Notebook 13 we used classical ML models -- Linear Regression and Random Forest -- with **hand-crafted features** extracted from SMILES strings (character counts, molecular weight, TPSA).

In this final notebook we ask: **what if the model could learn its own features from raw data?** That is the idea behind **deep learning**.

**Topics covered:**
- Neural network fundamentals (neuron, activation, layers, loss, training)
- A simple neural network on polymer features with PyTorch
- A CNN on molecular images (demo)
- Course recap and next steps

> This notebook is a **teaser**. Deep learning is a large field that deserves its own course. The goal here is to show you what is possible, not to make you an expert.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)

---
## 14.1 From Hand-Crafted Features to Learned Features

The pipeline we've been using:

```
SMILES  ->  [count_C, count_O, MolWt, TPSA, ...]  ->  ML model  ->  FFV prediction
                     ^
                     |
              we designed these features
```

The deep learning approach:

```
SMILES (or image, or graph)  ->  Neural network  ->  FFV prediction
                                       ^
                                       |
                              features are learned inside the network
```

The trade-off: deep learning usually needs **a lot of data** and **a lot of computing power**. With only a few thousand polymers, classical ML is often competitive or better.

---
## 14.2 A Single Neuron

A neuron computes:

$$\text{output} = \sigma(w_1 x_1 + w_2 x_2 + \ldots + w_n x_n + b)$$

This is exactly the linear model from Notebook 13, wrapped in a non-linear function $\sigma$ (the **activation function**).

The most common activation function is **ReLU**: $\text{ReLU}(x) = \max(0, x)$.

In [ ]:
x = np.linspace(-3, 3, 100)
relu = np.maximum(0, x)
sigmoid = 1 / (1 + np.exp(-x))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(x, relu); axes[0].set_title("ReLU"); axes[0].grid(alpha=0.3)
axes[1].plot(x, sigmoid); axes[1].set_title("Sigmoid"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 14.3 Stacking Neurons into Layers

Stack many neurons and many layers, and you get a **neural network**:

```
Input (6 features)
      |
[Linear -> ReLU]  <- hidden layer 1 (32 neurons)
      |
[Linear -> ReLU]  <- hidden layer 2 (16 neurons)
      |
[Linear]          <- output layer (1 prediction)
      |
  FFV prediction
```

**Training** a neural network means adjusting all the weights $w$ and biases $b$ to minimise a **loss function** (e.g. mean squared error) using an optimisation algorithm called **gradient descent**. You can think of it as walking downhill on the loss surface.

---
## 14.4 A Simple Neural Network on Polymer Features

Let's build the network above using **PyTorch**, the most widely used deep learning framework.

In [ ]:
# Load the same data as in Notebook 13
df = pd.read_csv("artifacts/ffv_ready_split.csv")
feature_cols = ["smiles_len", "count_star", "count_C", "count_O", "count_N", "count_equal"]

X_train = df.loc[df["split"]=="train", feature_cols].values.astype(np.float32)
y_train = df.loc[df["split"]=="train", "FFV"].values.astype(np.float32)
X_valid = df.loc[df["split"]=="valid", feature_cols].values.astype(np.float32)
y_valid = df.loc[df["split"]=="valid", "FFV"].values.astype(np.float32)

# Standardise (fit on train only)
mean, std = X_train.mean(axis=0), X_train.std(axis=0)
X_train_s = (X_train - mean) / std
X_valid_s = (X_valid - mean) / std

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train_s)
y_train_t = torch.tensor(y_train).unsqueeze(1)  # shape (N, 1)
X_valid_t = torch.tensor(X_valid_s)
y_valid_t = torch.tensor(y_valid).unsqueeze(1)

print("X_train tensor shape:", X_train_t.shape)
print("y_train tensor shape:", y_train_t.shape)

### Defining the network

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(42)
model = SimpleNN(n_features=X_train_t.shape[1])
print(model)

### Training loop

Training in PyTorch has a standard recipe:
1. Forward pass: compute predictions
2. Compute loss (how wrong are we?)
3. Backward pass: compute gradients
4. Update weights

In [ ]:
loss_fn = nn.MSELoss()                                   # mean squared error
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 200
train_losses, valid_losses = [], []

for epoch in range(n_epochs):
    # --- training step ---
    model.train()
    optimizer.zero_grad()
    preds = model(X_train_t)
    loss = loss_fn(preds, y_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    # --- validation ---
    model.eval()
    with torch.no_grad():
        valid_loss = loss_fn(model(X_valid_t), y_valid_t).item()
    valid_losses.append(valid_loss)

    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch+1:3d} | train loss {loss.item():.5f} | valid loss {valid_loss:.5f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="Train loss")
plt.plot(valid_losses, label="Valid loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Training curve")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate
from sklearn.metrics import r2_score, mean_absolute_error

model.eval()
with torch.no_grad():
    y_pred_nn = model(X_valid_t).numpy().flatten()

r2_nn = r2_score(y_valid, y_pred_nn)
mae_nn = mean_absolute_error(y_valid, y_pred_nn)

print(f"Neural Network on polymer features:")
print(f"  R²:  {r2_nn:.4f}")
print(f"  MAE: {mae_nn:.4f}")
print("\n(Compare to Random Forest R² from Notebook 13)")

> With only 6 features and ~5600 training samples, a neural network is **hard to beat a Random Forest**. Deep learning shines when you have **lots of data** and **rich, raw inputs** (images, text, sequences).

---
## 14.5 (Demo) A CNN on Molecular Images

Since Notebook 12 taught us how to turn SMILES into images, let's see a **Convolutional Neural Network (CNN)** process those images. A CNN learns spatial patterns (edges, rings, functional groups) from the image directly.

> This section is a **demo only** -- we train on a small batch, and the results will not be impressive. The point is to show you the architecture, not to win the Kaggle competition.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

def smiles_to_grayscale(smiles, size=64):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.ones((size, size), dtype=np.float32) * 255
    img = Draw.MolToImage(mol, size=(size, size))
    arr = np.array(img)[:, :, :3]
    return np.mean(arr, axis=2).astype(np.float32)

In [ ]:
# Build a small image dataset (use a subset so this runs in reasonable time)
N_TRAIN = 400
N_VALID = 100
IMG_SIZE = 64

train_df = df[df["split"] == "train"].sample(N_TRAIN, random_state=42)
valid_df = df[df["split"] == "valid"].sample(N_VALID, random_state=42)

print("Generating molecular images...")
X_img_train = np.stack([smiles_to_grayscale(s, IMG_SIZE) for s in train_df["SMILES"]])
X_img_valid = np.stack([smiles_to_grayscale(s, IMG_SIZE) for s in valid_df["SMILES"]])
y_img_train = train_df["FFV"].values.astype(np.float32)
y_img_valid = valid_df["FFV"].values.astype(np.float32)

# Normalise to [0, 1] and add a channel dimension (N, 1, H, W)
X_img_train = (X_img_train / 255.0)[:, None, :, :].astype(np.float32)
X_img_valid = (X_img_valid / 255.0)[:, None, :, :].astype(np.float32)

print("Train images shape:", X_img_train.shape)
print("Valid images shape:", X_img_valid.shape)

### CNN architecture

```
Image (1, 64, 64)
      |
[Conv2D (16 filters, 3x3) -> ReLU -> MaxPool]
      |
[Conv2D (32 filters, 3x3) -> ReLU -> MaxPool]
      |
[Flatten]
      |
[Linear -> ReLU]
      |
[Linear]          <- output (1 = predicted FFV)
```

A **Convolution** slides a small filter (e.g. 3x3) across the image, detecting local patterns. **MaxPool** downsamples by taking the max in small regions. Multiple conv-pool layers let the network build a hierarchy: edges -> shapes -> functional groups.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 64 -> 32
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 32 -> 16
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 16 * 16, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.head(self.features(x))


torch.manual_seed(42)
cnn = SimpleCNN()
print(cnn)

In [ ]:
X_img_train_t = torch.tensor(X_img_train)
y_img_train_t = torch.tensor(y_img_train).unsqueeze(1)
X_img_valid_t = torch.tensor(X_img_valid)
y_img_valid_t = torch.tensor(y_img_valid).unsqueeze(1)

optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

print("Training CNN (this may take a minute)...")
for epoch in range(30):
    cnn.train()
    optimizer.zero_grad()
    loss = loss_fn(cnn(X_img_train_t), y_img_train_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        cnn.eval()
        with torch.no_grad():
            valid_loss = loss_fn(cnn(X_img_valid_t), y_img_valid_t).item()
        print(f"Epoch {epoch+1:2d} | train {loss.item():.5f} | valid {valid_loss:.5f}")

In [ ]:
cnn.eval()
with torch.no_grad():
    y_pred_cnn = cnn(X_img_valid_t).numpy().flatten()

r2_cnn = r2_score(y_img_valid, y_pred_cnn)
print(f"CNN validation R²: {r2_cnn:.4f}")
print("(Expected: poor. We trained on only 400 images for 30 epochs!)")

> With 7000+ images, data augmentation, and more training time, a CNN could become competitive. But even better results on molecular property prediction are obtained with **Graph Neural Networks** (GNNs) that operate directly on the molecular graph -- that is the state of the art. This would be a topic for a follow-up course!

---
## 14.6 Course Recap

Over this course, you learned the complete data science toolkit:

| Notebooks | Topic | Polymer application |
|-----------|-------|---------------------|
| 00-03 | Python basics | Variables, control flow, functions |
| 02, 04 | Pandas | Loading and exploring `train.csv` |
| 05, 06, 11 | NumPy | Feature matrices, broadcasting, axes |
| 08 | Strings & regex | Parsing SMILES |
| 04 | Plotting | Visualising distributions |
| 10 | Linux | Command line for data science |
| 12 | RDKit & images | SMILES to molecular structures |
| 13 | Machine learning | Predicting FFV with sklearn |
| 14 | Deep learning | Neural networks, CNNs |

**Every research project in computational chemistry follows some version of this pipeline.**